# The Accelerated Backends

pycalphad ships two optional compute backends that generate C/CUDA code from the
symbolic `Model` energy expressions at runtime, compile it once per system
(disk-cached), and run batched equilibrium and property calculations through it:

- **`'c++'`** — compiles with the system C++ compiler and runs on a single CPU core.
  Install with `pip install pycalphad[cpp]`.
- **`'gpu'`** — runs the same generated kernels through CuPy on NVIDIA (CUDA) or
  AMD (ROCm) GPUs. Install with `pip install pycalphad[gpu]`. No CUDA toolkit
  install is required: when `nvcc` is absent, kernels compile through NVRTC,
  which the CuPy wheel already ships.

Three things to know before using them:

1. **Selection is explicit.** Nothing changes until you call
   `pycalphad.set_backend(...)` (global) or enter a `pycalphad.backend(...)`
   context. The default solver is untouched.
2. **Unsupported problems fall back silently.** The dispatch layer checks every
   call; anything the backend cannot handle (e.g. MQMQA models, phase-local
   conditions, custom solvers) runs on the reference solver instead, so enabling
   a backend never changes *which* problems you can solve.
3. **The first call on a new system compiles kernels** (seconds to ~a minute,
   depending on the system's complexity). Compiled kernels are cached on disk,
   so this cost is paid once per (database, phases, model) combination.

This notebook shows the selection API and a first side-by-side comparison on the
Al-Zn system. See `BACKENDS.md` in the repository root for the dependency and
tuning reference.

In [1]:
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
import pycalphad
from pycalphad import Database, Workspace, calculate, equilibrium, variables as v

# The backend to demonstrate: 'c++' runs generated kernels on the CPU,
# 'gpu' runs the same kernels through CuPy on an NVIDIA or AMD GPU.
BACKEND = 'c++'

def timed(label, fn):
    """Run fn(), print and return (result, elapsed seconds)."""
    start = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - start
    print(f'{label}: {elapsed:.1f} s')
    return result, elapsed


## A first comparison: Al-Zn

A 900-point temperature-composition equilibrium grid, run identically on the
reference solver and on the accelerated backend.

In [2]:
dbf = Database('../databases/alzn_mey.tdb')
comps = ['AL', 'ZN', 'VA']
phases = ['FCC_A1', 'HCP_A3', 'LIQUID']
conditions = {v.N: 1, v.P: 101325,
              v.T: (300, 1000, 24),
              v.X('ZN'): (0.02, 0.98, 0.033)}

In [3]:
# First accelerated call on a new system compiles and caches the kernels;
# do it once on a single point so the timings below reflect steady-state use.
with pycalphad.backend(BACKEND):
    equilibrium(dbf, comps, phases, {v.N: 1, v.P: 101325, v.T: 600, v.X('ZN'): 0.5});

In [4]:
ref, t_ref = timed('reference solver   ', lambda: equilibrium(dbf, comps, phases, conditions))

with pycalphad.backend(BACKEND):
    acc, t_acc = timed('accelerated backend', lambda: equilibrium(dbf, comps, phases, conditions))

print(f'speedup: {t_ref / t_acc:.1f}x')

reference solver   : 0.8 s


accelerated backend: 0.1 s
speedup: 7.6x


## Checking agreement

The backends are validated against the reference solver: same stable phase sets,
with Gibbs energies agreeing to numerical precision. (The compiled linear algebra
replaces LAPACK, so results are not bit-identical — agreement is at the
epsilon-times-condition-number scale of the underlying problem.)

In [5]:
def compare_results(ref, acc):
    """Agreement report between a reference and an accelerated result."""
    gm_ref = np.asarray(ref.GM.values, dtype=float).reshape(-1)
    gm_acc = np.asarray(acc.GM.values, dtype=float).reshape(-1)
    both = ~np.isnan(gm_ref) & ~np.isnan(gm_acc)
    print(f'conditions: {gm_ref.size}   converged: reference {int((~np.isnan(gm_ref)).sum())}, '
          f'accelerated {int((~np.isnan(gm_acc)).sum())}')
    dgm = np.abs(gm_acc[both] - gm_ref[both])
    print(f'max |dGM| over co-converged conditions: {dgm.max():.3e} J/mol '
          f'(relative: {(dgm / np.abs(gm_ref[both])).max():.3e})')
    ph_ref = np.asarray(ref.Phase.values).reshape(-1, ref.Phase.values.shape[-1])
    ph_acc = np.asarray(acc.Phase.values).reshape(-1, acc.Phase.values.shape[-1])
    mismatched = sum(sorted(p for p in ph_ref[i] if p) != sorted(p for p in ph_acc[i] if p)
                     for i in np.flatnonzero(both))
    print(f'stable-phase-set mismatches: {mismatched} of {int(both.sum())}')


In [6]:
compare_results(ref, acc)

conditions: 900   converged: reference 900, accelerated 900
max |dGM| over co-converged conditions: 4.327e-06 J/mol (relative: 2.638e-10)
stable-phase-set mismatches: 0 of 900


## The three ways to select a backend

```python
# 1. Globally, for everything that follows:
pycalphad.set_backend('c++')

# 2. Scoped, restoring the previous backend on exit:
with pycalphad.backend('c++'):
    eq = equilibrium(...)

# 3. Back to the reference solver:
pycalphad.set_backend('default')
```

`Workspace` (and everything built on it) dispatches through the same layer, so a
globally selected backend accelerates `wks.eq` too:

In [7]:
pycalphad.set_backend(BACKEND)
wks = Workspace(dbf, comps, phases, conditions)
print(f'GM at first grid point: {float(np.asarray(wks.eq.GM).reshape(-1)[0]):.2f} J/mol')
pycalphad.set_backend('default')

GM at first grid point: -8589.83 J/mol


## What is supported

Fully-determined condition grids over `N` / `P` / `T` with mole fractions
(`v.X`, including dilute and exactly-zero values), mass fractions (`v.W`),
chemical potentials (`v.MU`), and linear combinations of composition variables;
scalar and array fit parameters (`parameters=`); symbolic output properties
(`output='HM'`, ...); `equilibrium()`, `calculate()`, and `Workspace`.

Everything else — MQMQA/associate models with quadruplets, phase-local
conditions, custom solvers — falls back to the reference solver automatically.

The rest of this folder:

- **2 — Large-scale binary (Au-Bi)** and **3 — Large-scale ternary (Al-Cu-Fe)**:
  minute-scale reference workloads and their accelerated equivalents.
- **4 — Hard problems from the test suite**: the solver test suite's documented
  difficult cases, run as-is on both paths.
- **5 — Properties, conditions and Workspace**: beyond `GM`.
- **6 — Phase diagram mapping**: grid-based `binplot`/`ternplot`.
- **7 — Parameter sweeps and uncertainty**: many parameter sets in one call.